In [1]:
# =========================================
# 📦 CHECK & INSTALL PACKAGES AND FUNCTIONS
# =========================================

import importlib

def is_installed(pkg):
    return importlib.util.find_spec(pkg) is not None

required_packages = {
    "transformers": "transformers",
    "accelerate": "accelerate",
    "Bio": "biopython",
    "py3Dmol": "py3Dmol"
}

missing = [pip_name for mod, pip_name in required_packages.items() if not is_installed(mod)]

if missing:
    print(f"[*] Installing missing packages: {missing}")
    !pip install -q {' '.join(missing)}
else:
    print("[✓] All required Python packages already installed")

# ================================
# 🔧 CHECK & INSTALL FPOCKET
# ================================

import os
import shutil
from pathlib import Path
import subprocess

fpocket_path = shutil.which("fpocket")

if fpocket_path:
    print(f"[✓] fpocket already available at: {fpocket_path}")
else:
    print("[*] fpocket not found — installing from source...")

    if not Path("fpocket").exists():
        !git clone --depth 1 https://github.com/Discngine/fpocket.git

    %cd fpocket
    !make

    # Add to PATH
    os.environ["PATH"] += ":" + os.getcwd() + "/bin"

    %cd ..

    fpocket_path = shutil.which("fpocket")

# Verify fpocket works
if fpocket_path:
    result = subprocess.run(
        ["fpocket", "-h"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    if result.returncode == 0:
        print("[✓] fpocket is working correctly")
    else:
        print("[!] fpocket found but may not run correctly")
else:
    print("[✗] fpocket installation failed")

# ================================
# 🧠 LOAD ESMFOLD MODEL
# ================================

import torch
from transformers import AutoTokenizer, EsmForProteinFolding
from accelerate.test_utils.testing import get_backend

DEVICE, _, _ = get_backend()
print(f"[*] Using device: {DEVICE}")

print("[*] Loading ESMFold model (will download only first time)...")

tokenizer = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
model = EsmForProteinFolding.from_pretrained(
    "facebook/esmfold_v1",
    low_cpu_mem_usage=True
).to(DEVICE)

# Memory optimization
if DEVICE == "cuda":
    model.esm = model.esm.half()
    torch.backends.cuda.matmul.allow_tf32 = True

model.trunk.set_chunk_size(64)

print("[✓] ESMFold ready")

[✓] All required Python packages already installed
[*] fpocket not found — installing from source...
/mnt/disk3/Arun_work/esmfold/fpocket
cd src/qhull/ && make
make[1]: Entering directory '/mnt/disk3/Arun_work/esmfold/fpocket/src/qhull'
mkdir -p bin lib
make[1]: Leaving directory '/mnt/disk3/Arun_work/esmfold/fpocket/src/qhull'
/mnt/disk3/Arun_work/esmfold
[✓] fpocket is working correctly


/home/arunraj/miniconda3/envs/phosbind_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[*] Using device: cpu
[*] Loading ESMFold model (will download only first time)...


Loading weights: 100%|██████████| 4533/4533 [00:00<00:00, 9524.03it/s]
EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status     | 
-----------------------------------+------------+-
esm.embeddings.position_ids        | UNEXPECTED | 
esm.contact_head.regression.bias   | MISSING    | 
esm.contact_head.regression.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[✓] ESMFold ready


In [2]:
# ================================
# 🔧 USER INPUT
# ================================

from pathlib import Path
import re

USER_SEQUENCE = "MSKVCIIAWVYGRVQGVGFRYTTQYEAKRLGLTGYAKNLDDGSVEVVACGEEGQVEKLMQWLKSGGPRSARVERVLSEPHHPSGELTDFRIRLEHHHHHH"
JOB_NAME = "9SV1_1"


BASE_OUTPUT_DIR = f"./{JOB_NAME}_metadata"
USER_SEQUENCE = re.sub(r"\s+", "", USER_SEQUENCE).upper()
if not USER_SEQUENCE:
    raise ValueError("USER_SEQUENCE is empty.")

valid_aa = set("ACDEFGHIKLMNPQRSTVWY")
invalid_chars = sorted(set(USER_SEQUENCE) - valid_aa)

if invalid_chars:
    raise ValueError(f"Invalid amino acids found in sequence: {invalid_chars}")

JOB_NAME = JOB_NAME.strip()
if not JOB_NAME:
    raise ValueError("JOB_NAME cannot be empty.")
SAFE_JOB_NAME = re.sub(r"[^A-Za-z0-9_.-]", "_", JOB_NAME)

# Create output directory
JOB_DIR = Path(BASE_OUTPUT_DIR) / SAFE_JOB_NAME
JOB_DIR.mkdir(parents=True, exist_ok=True)

# Summary
print(f"[+] Job name      : {SAFE_JOB_NAME}")
print(f"[+] Sequence length: {len(USER_SEQUENCE)} aa")
print(f"[+] Output folder : {JOB_DIR}")

[+] Job name      : 9SV1_1
[+] Sequence length: 100 aa
[+] Output folder : 9SV1_1_metadata/9SV1_1


In [3]:
# ================================
# 🧰 HELPER FUNCTIONS
# ================================

import re
import json
import subprocess
from pathlib import Path

from transformers.models.esm.openfold_utils.protein import to_pdb, Protein as OFProtein
from transformers.models.esm.openfold_utils.feats import atom14_to_atom37


def convert_outputs_to_pdb(outputs):
    """
    Convert ESMFold model outputs to PDB strings.
    Returns a list of PDB strings, one per input sequence.
    """
    final_atom_positions = atom14_to_atom37(outputs["positions"][-1], outputs)

    outputs_np = {
        k: v.detach().cpu().numpy() if torch.is_tensor(v) else v
        for k, v in outputs.items()
    }

    final_atom_positions = final_atom_positions.detach().cpu().numpy()
    final_atom_mask = outputs_np["atom37_atom_exists"]

    pdbs = []
    for i in range(outputs_np["aatype"].shape[0]):
        pred = OFProtein(
            aatype=outputs_np["aatype"][i],
            atom_positions=final_atom_positions[i],
            atom_mask=final_atom_mask[i],
            residue_index=outputs_np["residue_index"][i] + 1,
            b_factors=outputs_np["plddt"][i],
            chain_index=outputs_np["chain_index"][i] if "chain_index" in outputs_np else None,
        )
        pdbs.append(to_pdb(pred))

    return pdbs


def save_pdb_files(pdb_strings, out_dir, prefix):
    """
    Save PDB strings to files.
    Returns a list of saved file paths.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    saved_paths = []
    for i, pdb_str in enumerate(pdb_strings):
        pdb_path = out_dir / f"{prefix}_{i}.pdb"
        pdb_path.write_text(pdb_str)
        saved_paths.append(pdb_path)
        print(f"[+] Saved PDB: {pdb_path}")

    return saved_paths


def run_fpocket(pdb_path):
    """
    Run fpocket on a PDB file.
    Returns the fpocket output directory.
    """
    pdb_path = Path(pdb_path).resolve()

    result = subprocess.run(
        ["fpocket", "-f", str(pdb_path)],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(f"fpocket failed for {pdb_path.name}")

    out_dir = pdb_path.parent / f"{pdb_path.stem}_out"
    print(f"[+] fpocket completed: {out_dir}")
    return out_dir


def parse_pocket_summary(out_dir):
    """
    Parse fpocket *_info.txt summary file.
    Returns a list of pocket dictionaries.
    """
    out_dir = Path(out_dir)
    info_files = list(out_dir.glob("*_info.txt"))

    if not info_files:
        print("[!] No fpocket summary file found.")
        return []

    info_file = info_files[0]
    pockets = []
    current = None

    with open(info_file, "r") as f:
        for line in f:
            line = line.rstrip()

            pocket_match = re.match(r"Pocket\s+(\d+)\s*:", line)
            if pocket_match:
                if current is not None:
                    pockets.append(current)
                current = {"id": int(pocket_match.group(1))}
                continue

            kv_match = re.match(r"\s+(.+?)\s*:\s*(.+)", line)
            if kv_match and current is not None:
                key = kv_match.group(1).strip()
                value = kv_match.group(2).strip()
                current[key] = value

    if current is not None:
        pockets.append(current)

    return pockets


def find_pocket_pdb_files(out_dir):
    """
    Find pocket atom PDB files produced by fpocket.
    Returns a dict: pocket_id -> file path
    """
    out_dir = Path(out_dir)
    pockets_dir = out_dir / "pockets"

    pocket_files = {}
    if pockets_dir.exists():
        for f in sorted(pockets_dir.glob("pocket*_atm.pdb")):
            match = re.search(r"pocket(\d+)_atm\.pdb", f.name)
            if match:
                pocket_id = int(match.group(1))
                pocket_files[pocket_id] = f

    return pocket_files


def save_json(data, out_file):
    """
    Save a Python object as JSON.
    """
    out_file = Path(out_file)
    with open(out_file, "w") as f:
        json.dump(data, f, indent=2)
    print(f"[+] Saved JSON: {out_file}")

In [4]:
# ================================
# 🔬 RUN ESMFOLD
# ================================

# Tokenize input sequence
tokenized_input = tokenizer(
    [USER_SEQUENCE],
    return_tensors="pt",
    add_special_tokens=False
)["input_ids"].to(DEVICE)

print(f"[*] Running ESMFold for job: {SAFE_JOB_NAME}")
print(f"[*] Sequence length: {len(USER_SEQUENCE)} aa")

# Run structure prediction
with torch.no_grad():
    output = model(tokenized_input)

# Basic confidence summary
mean_plddt = output["plddt"].mean().item()
min_plddt = output["plddt"].min().item()
max_plddt = output["plddt"].max().item()

print("[✓] ESMFold prediction complete")
print(f"[+] Mean pLDDT: {mean_plddt:.2f}")
print(f"[+] Min  pLDDT: {min_plddt:.2f}")
print(f"[+] Max  pLDDT: {max_plddt:.2f}")

if mean_plddt < 0.90:
    print("[!] Warning: overall structure confidence is low; pocket predictions may be less reliable. Consider using a pdb file from a high-quality source.")

[*] Running ESMFold for job: 9SV1_1
[*] Sequence length: 100 aa
[✓] ESMFold prediction complete
[+] Mean pLDDT: 0.76
[+] Min  pLDDT: 0.24
[+] Max  pLDDT: 0.96
[!] Warning: overall structure confidence is low; pocket predictions may be less reliable. Consider using a pdb file from a high-quality source.


In [5]:
# ================================
# 💾 CONVERT OUTPUT TO PDB AND SAVE
# ================================

# Convert ESMFold outputs to PDB strings
pdb_strings = convert_outputs_to_pdb(output)

# Save PDB files
pdb_paths = save_pdb_files(
    pdb_strings=pdb_strings,
    out_dir=JOB_DIR,
    prefix=SAFE_JOB_NAME
)

print(f"[✓] Saved {len(pdb_paths)} PDB file(s)")

[+] Saved PDB: 9SV1_1_metadata/9SV1_1/9SV1_1_0.pdb
[✓] Saved 1 PDB file(s)


In [ ]:
# ================================
# 🧪 FPOCKET + 5 Å BINDING-SITE RESIDUES
# ================================

from pathlib import Path
import re
import math
import subprocess

DIST_CUTOFF = 5.0

AA3_TO_AA1 = {
    "ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C",
    "GLU":"E","GLN":"Q","GLY":"G","HIS":"H","ILE":"I",
    "LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P",
    "SER":"S","THR":"T","TRP":"W","TYR":"Y","VAL":"V"
}

# --------------------------------
# Run fpocket
# --------------------------------
def run_fpocket(pdb_path):
    pdb_path = Path(pdb_path).resolve()

    result = subprocess.run(
        ["fpocket", "-f", str(pdb_path)],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(f"fpocket failed for {pdb_path.name}")

    return pdb_path.parent / f"{pdb_path.stem}_out"


# --------------------------------
# Parse fpocket summary
# --------------------------------
def parse_fpocket_info(info_file):
    pockets = {}
    current = None

    with open(info_file, "r") as f:
        for line in f:
            m = re.match(r"Pocket\s+(\d+)\s*:", line)
            if m:
                current = int(m.group(1))
                pockets[current] = {}
                continue

            if current is None:
                continue

            fields = [
                ("score", r"Score\s*:\s*(.+)"),
                ("druggability", r"Druggability Score\s*:\s*(.+)"),
                ("volume", r"Volume\s*:\s*(.+)"),
                ("n_spheres", r"Number of alpha spheres\s*:\s*(.+)")
            ]

            for key, pattern in fields:
                hit = re.search(pattern, line)
                if hit:
                    val = hit.group(1).strip()
                    try:
                        pockets[current][key] = int(val) if key == "n_spheres" else float(val)
                    except ValueError:
                        pockets[current][key] = val

    return pockets


# --------------------------------
# Find pocket files
# --------------------------------
def find_pocket_files(out_dir):
    pocket_dir = Path(out_dir) / "pockets"
    files = {}

    if not pocket_dir.exists():
        return files

    for p in pocket_dir.glob("pocket*_atm.pdb"):
        m = re.search(r"pocket(\d+)_atm\.pdb", p.name)
        if m:
            pid = int(m.group(1))
            files[pid] = {
                "atm": p,
                "vert": pocket_dir / f"pocket{pid}_vert.pqr"
            }

    return files


# --------------------------------
# Read protein atoms
# --------------------------------
def parse_protein_atoms(pdb_file):
    atoms = []

    with open(pdb_file, "r") as f:
        for line in f:
            if not line.startswith("ATOM"):
                continue

            try:
                atoms.append({
                    "resname": line[17:20].strip(),
                    "chain": line[21].strip() or "-",
                    "resid": int(line[22:26].strip()),
                    "coord": (
                        float(line[30:38]),
                        float(line[38:46]),
                        float(line[46:54])
                    )
                })
            except ValueError:
                continue

    return atoms


# --------------------------------
# Read alpha sphere coordinates
# --------------------------------
def get_alpha_sphere_coords(vert_file):
    coords = []

    if not Path(vert_file).exists():
        return coords

    with open(vert_file, "r") as f:
        for line in f:
            if not line.startswith(("ATOM", "HETATM")):
                continue
            try:
                coords.append((
                    float(line[30:38]),
                    float(line[38:46]),
                    float(line[46:54])
                ))
            except ValueError:
                continue

    return coords


# --------------------------------
# Distance
# --------------------------------
def distance(a, b):
    return math.sqrt(
        (a[0] - b[0])**2 +
        (a[1] - b[1])**2 +
        (a[2] - b[2])**2
    )


# --------------------------------
# Residues within cutoff of alpha spheres
# --------------------------------
def get_binding_site_residues(protein_pdb, vert_file, cutoff=5.0):
    protein_atoms = parse_protein_atoms(protein_pdb)
    alpha_coords = get_alpha_sphere_coords(vert_file)

    selected = {}

    if not alpha_coords:
        return []

    for atom in protein_atoms:
        min_dist = min(distance(atom["coord"], ac) for ac in alpha_coords)

        if min_dist <= cutoff:
            key = (atom["chain"], atom["resid"], atom["resname"])

            if key not in selected or min_dist < selected[key]["min_dist"]:
                selected[key] = {
                    "index": atom["resid"],
                    "resid1": atom["resid"],
                    "chain": atom["chain"],
                    "resname": atom["resname"],
                    "min_dist": min_dist
                }

    residues = sorted(selected.values(), key=lambda x: x["index"])
    return residues


# --------------------------------
# Rank pockets
# --------------------------------
def rank_pockets(pocket_info):
    def safe_float(x, default=float("-inf")):
        try:
            return float(x)
        except (TypeError, ValueError):
            return default

    ranked = sorted(
        pocket_info.items(),
        key=lambda kv: (
            safe_float(kv[1].get("score")),
            safe_float(kv[1].get("druggability")),
            safe_float(kv[1].get("n_spheres"))
        ),
        reverse=True
    )
    return ranked


# --------------------------------
# Main
# --------------------------------
all_binding_sites = {}

for pdb_path in pdb_paths:
    pdb_path = Path(pdb_path)

    print(f"\n{'='*70}")
    print(f"Processing: {pdb_path.name}")
    print(f"{'='*70}")

    out_dir = run_fpocket(pdb_path)
    info_file = out_dir / f"{pdb_path.stem}_info.txt"

    if not info_file.exists():
        print("[!] fpocket info file not found")
        continue

    pocket_info = parse_fpocket_info(info_file)
    pocket_files = find_pocket_files(out_dir)

    if not pocket_info or not pocket_files:
        print("[!] No pockets found")
        continue

    ranked = rank_pockets(pocket_info)
    all_binding_sites[str(pdb_path)] = []

    for display_idx, (pid, info) in enumerate(ranked, start=1):
        if pid not in pocket_files:
            continue

        residues = get_binding_site_residues(
            protein_pdb=pdb_path,
            vert_file=pocket_files[pid]["vert"],
            cutoff=DIST_CUTOFF
        )

        residue_indices = [r["index"] for r in residues]
        residue_names = [r["resname"] for r in residues]

        all_binding_sites[str(pdb_path)].append({
            "ranked_pocket_number": display_idx,
            "fpocket_id": pid,
            "score": info.get("score"),
            "druggability": info.get("druggability"),
            "volume": info.get("volume"),
            "residue_indices": residue_indices,
            "residue_names": residue_names
        })

        print(f"\nPocket {display_idx}")
        #print(f"fpocket ID: {pid}")
        #print(f"Score: {info.get('score', 'N/A')}")
        #print(f"Druggability Score: {info.get('druggability', 'N/A')}")
        #print(f"Volume: {info.get('volume', 'N/A')}")
        print(f"Binding-site residue indices: {residue_indices}")
        print(f"Binding-site residue names: {residue_names}")


Processing: 9SV1_1_0.pdb

Pocket 1
Binding-site residue indices: [4, 5, 6, 52, 55, 56, 59, 79]
Binding-site residue names: ['VAL', 'CYS', 'ILE', 'GLU', 'VAL', 'GLU', 'MET', 'PRO']

Pocket 2
Binding-site residue indices: [18, 21, 22, 25, 26, 61, 66, 67]
Binding-site residue names: ['GLY', 'TYR', 'THR', 'TYR', 'GLU', 'TRP', 'GLY', 'PRO']

Pocket 3
Binding-site residue indices: [3, 4, 5, 31, 33, 49, 50, 51, 82, 84, 85, 86]
Binding-site residue names: ['LYS', 'VAL', 'CYS', 'GLY', 'THR', 'CYS', 'GLY', 'GLU', 'PRO', 'GLY', 'GLU', 'LEU']

Pocket 4
Binding-site residue indices: [1, 2, 51, 52, 53, 54]
Binding-site residue names: ['MET', 'SER', 'GLU', 'GLU', 'GLY', 'GLN']

Pocket 5
Binding-site residue indices: [3, 4, 5, 79, 80, 81, 82]
Binding-site residue names: ['LYS', 'VAL', 'CYS', 'PRO', 'HIS', 'HIS', 'PRO']

Pocket 6
Binding-site residue indices: [34, 35, 85, 86, 87, 88, 90, 91, 92]
Binding-site residue names: ['GLY', 'TYR', 'GLU', 'LEU', 'THR', 'ASP', 'ARG', 'ILE', 'ARG']

Pocket 7
Bindi

In [10]:
# ================================
# 🧠 EACH POCKET EMBEDDING AS SEPARATE VARIABLE
# ================================

import re
import math
import subprocess
import importlib
from pathlib import Path
import torch

DIST_CUTOFF = 5.0
TOP_K_POCKETS = 10   # number of pockets to keep

POSITIVE_RES = {"ARG", "LYS", "HIS"}
POLAR_RES = {"SER", "THR", "ASN", "GLN", "CYS", "TYR"}
NEGATIVE_RES = {"ASP", "GLU"}
HYDROPHOBIC_RES = {"ALA", "VAL", "ILE", "LEU", "MET", "PHE", "TRP", "PRO"}

W_POSITIVE = 3.0
W_POLAR = 1.5
W_NEGATIVE = -3.0
W_HYDROPHOBIC = -1.0

# --------------------------------
# Install/load esm
# --------------------------------
if importlib.util.find_spec("esm") is None:
    print("[*] Installing esm package...")
    get_ipython().system("pip install -q fair-esm")

import esm

# --------------------------------
# Load ESM2
# --------------------------------
esm_model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
batch_converter = alphabet.get_batch_converter()

esm_model.eval()
if torch.cuda.is_available():
    esm_model = esm_model.cuda()

# --------------------------------
# Your embedding functions
# --------------------------------
def get_set_of_embeddings_at_binding_site_esm(model, alphabet, batch_converter, seq_data, indices_for_averaging):
    batch_labels, batch_strs, batch_tokens = batch_converter([(seq_data[0], seq_data[1])])

    if torch.cuda.is_available():
        batch_tokens = batch_tokens.cuda()

    with torch.no_grad():
        results = model(batch_tokens, repr_layers=[33], return_contacts=False)

    token_representations = results["representations"][33]
    return [token_representations[0][i] for i in indices_for_averaging]


def preprocess_sequence(sequence, important_residues, esm_model, alphabet, batch_converter):
    set_of_embeddings = get_set_of_embeddings_at_binding_site_esm(
        esm_model, alphabet, batch_converter, ("protein", sequence), important_residues
    )
    avg_embedding = torch.mean(torch.stack(set_of_embeddings), dim=0)
    return avg_embedding.detach().cpu().numpy()

# --------------------------------
# fpocket helpers
# --------------------------------
def run_fpocket(pdb_path):
    pdb_path = Path(pdb_path).resolve()
    result = subprocess.run(
        ["fpocket", "-f", str(pdb_path)],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(f"fpocket failed for {pdb_path.name}")
    return pdb_path.parent / f"{pdb_path.stem}_out"


def parse_fpocket_info(info_file):
    pockets = {}
    current = None

    with open(info_file, "r") as f:
        for line in f:
            m = re.match(r"Pocket\s+(\d+)\s*:", line)
            if m:
                current = int(m.group(1))
                pockets[current] = {}
                continue

            if current is None:
                continue

            fields = [
                ("score", r"Score\s*:\s*(.+)"),
                ("volume", r"Volume\s*:\s*(.+)"),
                ("n_spheres", r"Number of alpha spheres\s*:\s*(.+)")
            ]

            for key, pattern in fields:
                hit = re.search(pattern, line)
                if hit:
                    val = hit.group(1).strip()
                    try:
                        pockets[current][key] = int(val) if key == "n_spheres" else float(val)
                    except ValueError:
                        pockets[current][key] = val
    return pockets


def find_pocket_files(out_dir):
    pocket_dir = Path(out_dir) / "pockets"
    files = {}
    if not pocket_dir.exists():
        return files

    for p in pocket_dir.glob("pocket*_atm.pdb"):
        m = re.search(r"pocket(\d+)_atm\.pdb", p.name)
        if m:
            pid = int(m.group(1))
            files[pid] = {
                "atm": p,
                "vert": pocket_dir / f"pocket{pid}_vert.pqr"
            }
    return files


def parse_protein_atoms(pdb_file):
    atoms = []
    with open(pdb_file, "r") as f:
        for line in f:
            if not line.startswith("ATOM"):
                continue
            try:
                atoms.append({
                    "resname": line[17:20].strip(),
                    "chain": line[21].strip() or "-",
                    "resid": int(line[22:26].strip()),
                    "coord": (
                        float(line[30:38]),
                        float(line[38:46]),
                        float(line[46:54])
                    )
                })
            except ValueError:
                continue
    return atoms


def get_alpha_sphere_coords(vert_file):
    coords = []
    vert_file = Path(vert_file)
    if not vert_file.exists():
        return coords

    with open(vert_file, "r") as f:
        for line in f:
            if not line.startswith(("ATOM", "HETATM")):
                continue
            try:
                coords.append((
                    float(line[30:38]),
                    float(line[38:46]),
                    float(line[46:54])
                ))
            except ValueError:
                continue
    return coords


def distance(a, b):
    return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2 + (a[2]-b[2])**2)


def get_binding_site_residues(protein_pdb, vert_file, cutoff=5.0):
    protein_atoms = parse_protein_atoms(protein_pdb)
    alpha_coords = get_alpha_sphere_coords(vert_file)

    selected = {}
    if not alpha_coords:
        return []

    for atom in protein_atoms:
        min_dist = min(distance(atom["coord"], ac) for ac in alpha_coords)

        if min_dist <= cutoff:
            key = (atom["chain"], atom["resid"], atom["resname"])
            if key not in selected or min_dist < selected[key]["min_dist"]:
                seq_index = atom["resid"] - 1
                selected[key] = {
                    "index": seq_index,
                    "resname": atom["resname"],
                    "min_dist": min_dist
                }

    return sorted(selected.values(), key=lambda x: x["index"])


def pocket_residue_counts(residues):
    counts = {"positive": 0, "polar": 0, "negative": 0, "hydrophobic": 0, "other": 0}
    for r in residues:
        res = r["resname"]
        if res in POSITIVE_RES:
            counts["positive"] += 1
        elif res in POLAR_RES:
            counts["polar"] += 1
        elif res in NEGATIVE_RES:
            counts["negative"] += 1
        elif res in HYDROPHOBIC_RES:
            counts["hydrophobic"] += 1
        else:
            counts["other"] += 1
    return counts


def anion_preference_score(residues):
    counts = pocket_residue_counts(residues)
    score = (
        W_POSITIVE * counts["positive"] +
        W_POLAR * counts["polar"] +
        W_NEGATIVE * counts["negative"] +
        W_HYDROPHOBIC * counts["hydrophobic"]
    )
    return score, counts


def select_top_anion_pockets(pdb_path, pocket_info, pocket_files, cutoff=5.0, top_k=3):
    candidates = []

    for pid, info in pocket_info.items():
        if pid not in pocket_files:
            continue

        residues = get_binding_site_residues(
            protein_pdb=pdb_path,
            vert_file=pocket_files[pid]["vert"],
            cutoff=cutoff
        )

        pref_score, counts = anion_preference_score(residues)

        candidates.append({
            "pid": pid,
            "fpocket_score": info.get("score", None),
            "volume": info.get("volume", None),
            "n_spheres": info.get("n_spheres", None),
            "anion_pref_score": pref_score,
            "counts": counts,
            "residues": residues
        })

    def safe_float(x, default=float("-inf")):
        try:
            return float(x)
        except (TypeError, ValueError):
            return default

    candidates = sorted(
        candidates,
        key=lambda x: (
            x["anion_pref_score"],
            safe_float(x["fpocket_score"]),
            safe_float(x["n_spheres"])
        ),
        reverse=True
    )

    return candidates[:top_k]

# --------------------------------
# MAIN
# --------------------------------
pocket_embeddings = {}

for pdb_path in pdb_paths:
    pdb_path = Path(pdb_path)
    out_dir = run_fpocket(pdb_path)
    info_file = out_dir / f"{pdb_path.stem}_info.txt"

    pocket_info = parse_fpocket_info(info_file)
    pocket_files = find_pocket_files(out_dir)

    selected_pockets = select_top_anion_pockets(
        pdb_path=pdb_path,
        pocket_info=pocket_info,
        pocket_files=pocket_files,
        cutoff=DIST_CUTOFF,
        top_k=TOP_K_POCKETS
    )

    for rank_i, pocket in enumerate(selected_pockets, start=1):
        pocket_indices = [r["index"] for r in pocket["residues"]]
        pocket_resnames = [r["resname"] for r in pocket["residues"]]

        if not pocket_indices:
            continue

        emb = preprocess_sequence(
            USER_SEQUENCE,
            pocket_indices,
            esm_model,
            alphabet,
            batch_converter
        )

        var_name = f"{pdb_path.stem}_pocket_{rank_i}"
        pocket_embeddings[var_name] = {
            "embedding": emb,
            "indices": pocket_indices,
            "resnames": pocket_resnames,
            "fpocket_id": pocket["pid"],
            "fpocket_score": pocket["fpocket_score"],
            "anion_preference_score": pocket["anion_pref_score"]
        }

        print(f"{var_name}")
        print(f"  indices: {pocket_indices}")
        print(f"  residues: {pocket_resnames}")

9SV1_1_0_pocket_1
  indices: [2, 3, 4, 78, 79, 80, 81]
  residues: ['LYS', 'VAL', 'CYS', 'PRO', 'HIS', 'HIS', 'PRO']
9SV1_1_0_pocket_2
  indices: [19, 37, 91, 92, 93, 95]
  residues: ['ARG', 'ASN', 'ARG', 'LEU', 'GLU', 'HIS']
9SV1_1_0_pocket_3
  indices: [13, 14, 15, 16, 17, 18, 19, 20, 37]
  residues: ['VAL', 'GLN', 'GLY', 'VAL', 'GLY', 'PHE', 'ARG', 'TYR', 'ASN']
9SV1_1_0_pocket_4
  indices: [34, 46, 79, 81, 82, 83, 84, 85, 91]
  residues: ['TYR', 'VAL', 'HIS', 'PRO', 'SER', 'GLY', 'GLU', 'LEU', 'ARG']
9SV1_1_0_pocket_5
  indices: [33, 34, 84, 85, 86, 87, 89, 90, 91]
  residues: ['GLY', 'TYR', 'GLU', 'LEU', 'THR', 'ASP', 'ARG', 'ILE', 'ARG']
9SV1_1_0_pocket_6
  indices: [17, 20, 21, 24, 25, 60, 65, 66]
  residues: ['GLY', 'TYR', 'THR', 'TYR', 'GLU', 'TRP', 'GLY', 'PRO']
9SV1_1_0_pocket_7
  indices: [2, 3, 4, 30, 32, 48, 49, 50, 81, 83, 84, 85]
  residues: ['LYS', 'VAL', 'CYS', 'GLY', 'THR', 'CYS', 'GLY', 'GLU', 'PRO', 'GLY', 'GLU', 'LEU']
9SV1_1_0_pocket_8
  indices: [0, 1, 50, 51, 5

In [11]:
# ================================
# 🔮 PREDICT FOR EACH POCKET EMBEDDING
# ================================

import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---- same MLP definition (must match training) ----
class MLP(nn.Module):
    def __init__(self, in_dim, h1, h2, h3, out_dim, p_drop=0.3, use_bn=True):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, h1)
        self.bn1 = nn.BatchNorm1d(h1) if use_bn else nn.Identity()
        self.fc2 = nn.Linear(h1, h2)
        self.bn2 = nn.BatchNorm1d(h2) if use_bn else nn.Identity()
        self.fc3 = nn.Linear(h2, h3)
        self.bn3 = nn.BatchNorm1d(h3) if use_bn else nn.Identity()
        self.out = nn.Linear(h3, out_dim)
        self.drop = nn.Dropout(p_drop)

    def forward(self, x):
        x = self.fc1(x); x = self.bn1(x); x = F.relu(x)
        x = self.fc2(x); x = self.bn2(x); x = F.relu(x)
        x = self.fc3(x); x = self.bn3(x); x = F.relu(x)
        return self.out(x)

def load_mlp(save_dir, device=None):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

    with open(os.path.join(save_dir, "metadata.json"), "r") as f:
        meta = json.load(f)

    in_dim = meta["feature_dim"]
    out_dim = meta["n_classes"]

    hidden_sizes = meta["architecture"]["hidden_sizes"]
    h1, h2, h3 = hidden_sizes

    best_params = meta["best_params"]

    model = MLP(
        in_dim=in_dim,
        h1=h1,
        h2=h2,
        h3=h3,
        out_dim=out_dim,
        p_drop=best_params["dropout"],
        use_bn=best_params["use_bn"],
    ).to(device)

    sd = torch.load(os.path.join(save_dir, "mlp_state_dict.pt"), map_location=device)
    model.load_state_dict(sd)
    model.eval()
    return model, meta, device

def predict_one(vec_1d, model, meta, device, topk=3):
    x = torch.tensor(np.asarray(vec_1d, dtype=np.float32), device=device).view(1, -1)

    if x.shape[1] != meta["feature_dim"]:
        raise ValueError(f"Expected {meta['feature_dim']} features, got {x.shape[1]}")

    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

    pred_class = int(np.argmax(probs))
    pred_name = meta["class_id_to_name"].get(str(pred_class), str(pred_class))
    conf = float(probs[pred_class])

    print("\n==============================")
    print("PREDICTION RESULT (MLP)")
    print("==============================")
    print(f"Predicted Class ID : {pred_class}")
    print(f"Predicted Class    : {pred_name}")
    print(f"Confidence         : {conf:.4f}")

    print("\nFull Probability Distribution:")
    for i, p in enumerate(probs):
        cname = meta["class_id_to_name"].get(str(i), str(i))
        print(f"  Class {i} ({cname}): {p:.4f}")

    return pred_class, pred_name, conf, probs


# Load trained model
model, meta, device = load_mlp(
    "/mnt/disk3/Arun_work/phospher_work/Results/results_mlp_BW_hparam_sweep/best_model_20260320-103259"
)

all_predictions = {}

for name, data in pocket_embeddings.items():
    emb = data["embedding"]

    print(f"\n{'='*70}")
    print(f"Pocket: {name}")
    print(f"{'='*70}")
    print(f"fpocket ID: {data['fpocket_id']}")
    print(f"Residue indices: {data['indices']}")
    print(f"Residue names: {data['resnames']}")

    pred_class, pred_name, conf, probs = predict_one(
        emb,
        model,
        meta,
        device,
        topk=3
    )

    all_predictions[name] = {
        "fpocket_id": data["fpocket_id"],
        "indices": data["indices"],
        "resnames": data["resnames"],
        "predicted_class_id": pred_class,
        "predicted_class_name": pred_name,
        "confidence": conf,
        "probabilities": probs.tolist()
    }


Pocket: 9SV1_1_0_pocket_1
fpocket ID: 5
Residue indices: [2, 3, 4, 78, 79, 80, 81]
Residue names: ['LYS', 'VAL', 'CYS', 'PRO', 'HIS', 'HIS', 'PRO']

PREDICTION RESULT (MLP)
Predicted Class ID : 0
Predicted Class    : phosphate
Confidence         : 0.7295

Full Probability Distribution:
  Class 0 (phosphate): 0.7295
  Class 1 (sulfate): 0.2048
  Class 2 (chloride): 0.0654
  Class 3 (nitrate): 0.0001
  Class 4 (carbonate): 0.0003

Pocket: 9SV1_1_0_pocket_2
fpocket ID: 7
Residue indices: [19, 37, 91, 92, 93, 95]
Residue names: ['ARG', 'ASN', 'ARG', 'LEU', 'GLU', 'HIS']

PREDICTION RESULT (MLP)
Predicted Class ID : 2
Predicted Class    : chloride
Confidence         : 0.7145

Full Probability Distribution:
  Class 0 (phosphate): 0.0014
  Class 1 (sulfate): 0.2841
  Class 2 (chloride): 0.7145
  Class 3 (nitrate): 0.0001
  Class 4 (carbonate): 0.0000

Pocket: 9SV1_1_0_pocket_3
fpocket ID: 9
Residue indices: [13, 14, 15, 16, 17, 18, 19, 20, 37]
Residue names: ['VAL', 'GLN', 'GLY', 'VAL', 'GLY